# 03 · Models and Structured Output

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama. OpenAI cells run
> only if `OPENAI_API_KEY` is set in `02_OPENAI_AGENTS_SDK/.env`.

An agent is only as good as its model, and a program can only act on an
answer it can parse. This notebook covers both: how the SDK connects to models
(including local Ollama), how to tune them, and how to make an agent return a
**typed Python object** instead of text.

**You will learn**
- The three ways to give an agent a model, and the two OpenAI APIs behind them
- How `common.get_model()` connects the SDK to Ollama
- `ModelSettings`: temperature, max tokens and friends
- Different models for different agents in one system
- **`output_type`**: structured, validated results, and the trap where the model ignores your field descriptions
- Strict schemas, and counting tokens with `usage`

**Prerequisites:** notebooks 01–02.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

from typing import Literal
from pydantic import BaseModel, Field
from agents import Agent, Runner, ModelSettings, AgentOutputSchema, UserError
from common import get_model, has_key

model = get_model()

## 1 · Why models are pluggable

The SDK is written against an abstract **`Model`** interface: "given
instructions, input, tools and an output schema, return a response". Any
backend that implements it works, so you can develop against a free local model
and ship on a cloud model without touching agent code.

There are three ways to give an agent a model:

| You pass | What happens | Typical use |
|---|---|---|
| a **string**, e.g. `'gpt-4.1-mini'` | the default provider (OpenAI) resolves it | OpenAI models |
| a **`Model` object** | used as is, with whatever client and endpoint it wraps | Ollama, other OpenAI-compatible servers |
| `RunConfig(model_provider=...)` | a provider resolves every string in the run | switching a whole system at once |

**Two OpenAI APIs.** OpenAI offers the newer **Responses API** and the older,
widely copied **Chat Completions API**. The SDK has a model class for each:

- `OpenAIResponsesModel`: the default for OpenAI; supports OpenAI's hosted
  tools (web search, file search, code interpreter).
- `OpenAIChatCompletionsModel`: speaks Chat Completions, which **many other
  servers imitate**, including Ollama, vLLM and LM Studio. This is how we run locally.

## 2 · Connecting to Ollama

Ollama exposes an OpenAI-compatible endpoint at `http://localhost:11434/v1`.
Point an `AsyncOpenAI` client at it and wrap it in `OpenAIChatCompletionsModel`.
That's all `common.get_model()` does:

In [ ]:
import inspect, common
print(inspect.getsource(common.get_model))

Two more details from `common.py`:

- **Tracing export is off unless you have an OpenAI key.** By default the SDK
  uploads traces to OpenAI's dashboard. With a local model and no key that would
  just fail, so `common.py` calls `set_tracing_disabled(...)`. Notebook 10 shows
  how to record traces locally instead.
- **Ollama ignores the API key**, but the OpenAI client requires one, so we pass `'ollama'`.

Every provider you've configured, same agent definition:

In [ ]:
async def ask(m):
    agent = Agent(name='Explainer', instructions='Answer in exactly one sentence.', model=m)
    return (await Runner.run(agent, 'What is a context window?')).final_output

print('ollama :', await ask(get_model('ollama')))
print('openai :', await ask(get_model('openai')) if has_key('openai') else '(skipped: no OPENAI_API_KEY in .env)')

Other providers (Anthropic, Gemini, Mistral, …) can be reached through the
SDK's optional **LiteLLM** integration (`pip install "openai-agents[litellm]"`,
then `model='litellm/anthropic/claude-...'`), or any OpenAI-compatible gateway
with `OpenAIChatCompletionsModel`.

## 3 · `ModelSettings`: tuning the model

`model_settings=ModelSettings(...)` on an agent (or in `RunConfig`) controls
how the model is called:

| Setting | Meaning |
|---|---|
| `temperature` | randomness: 0 for routing, extraction and tests; higher for creative text |
| `max_tokens` | cap on the reply length |
| `tool_choice` | `'auto'`, `'required'`, `'none'` or a tool name (provider support varies: Ollama ignores it) |
| `parallel_tool_calls` | allow several tool calls in one turn |
| `extra_body` / `extra_args` | provider-specific options passed straight through |

In [ ]:
creative = Agent(name='Namer', model=model, instructions='Invent one product name. Reply with the name only.',
                 model_settings=ModelSettings(temperature=1.2))
precise = creative.clone(model_settings=ModelSettings(temperature=0))

print('temperature 1.2:', [ (await Runner.run(creative, 'A delivery robot for stairs.')).final_output for _ in range(3) ])
print('temperature 0  :', [ (await Runner.run(precise, 'A delivery robot for stairs.')).final_output for _ in range(3) ])

At temperature 0 you usually get the same answer every time; at 1.2 you get
variety. Neither is "better": use low temperatures where you need consistency
(classification, extraction, tool arguments) and higher ones where you want options.

**Different models per agent.** Each agent has its own `model`, so a system can
use a small fast model for routing and a strong one for writing. With one local
model we can't show the quality difference, but the code pattern is simply:

```python
router = Agent(name='Router', model=get_model('ollama'), ...)          # cheap, fast
writer = Agent(name='Writer', model=get_model('openai'), ...)          # strong
```

## 4 · Structured output with `output_type`

A program can't act on "I'd say this is probably a billing issue…". Give the
agent an **`output_type`** and the run ends only when the model produces a valid
object of that type. `result.final_output` is then that object:

In [ ]:
class Ticket(BaseModel):
    category: Literal['billing', 'technical', 'other']
    urgency: int = Field(ge=1, le=5)
    summary: str


RUBRIC = ('Classify the support ticket. Urgency: 1 = question, no impact; 3 = something is broken; '
          '5 = customer is losing money now or faces a deadline within 48 hours.')

classifier = Agent(name='Classifier', instructions=RUBRIC, model=model, output_type=Ticket)
result = await Runner.run(classifier, 'I was charged three times this month and my rent is due tomorrow!!')

ticket = result.final_output
print(type(ticket).__name__, ticket)
print('route on it:', 'escalate' if ticket.urgency >= 4 else 'queue', '→', ticket.category)

How it works: the SDK turns the type into a **JSON schema** and sends it with the
request, the provider constrains the model to produce matching JSON, and the
SDK validates the JSON with Pydantic before handing you the object. `Literal`
guarantees `category` is one of the three values, and `Field(ge=1, le=5)` keeps
urgency in range.

### The trap: the model may not read your field descriptions

Putting the urgency scale in a `Field(description=...)` looks cleaner than
writing it in the instructions. Watch what happens on Ollama:

In [ ]:
class TicketWithDescriptions(BaseModel):
    category: Literal['billing', 'technical', 'other']
    urgency: int = Field(ge=1, le=5, description=(
        '1 = question, no impact; 3 = something is broken; '
        '5 = customer is losing money now or faces a deadline within 48 hours'))

TEXT = 'I was charged three times this month and my rent is due tomorrow!!'
only_schema = Agent(name='A', instructions='Classify the support ticket.', model=model,
                    output_type=TicketWithDescriptions)
with_rubric = Agent(name='B', instructions=RUBRIC, model=model, output_type=TicketWithDescriptions)

for attempt in range(1, 3):
    a = (await Runner.run(only_schema, TEXT)).final_output.urgency
    b = (await Runner.run(with_rubric, TEXT)).final_output.urgency
    print(f'run {attempt}: scale only in the schema → urgency {a} | scale in the instructions → urgency {b}')

With the scale only in the schema, the urgency is wrong (and can vary between
runs). With the same scale in the instructions, it's 5 every time. On Ollama
the JSON schema **constrains the format of the output**, but it isn't shown to
the model as text it reasons about. So:

- **Format ≠ meaning.** Structured output guarantees a valid `Ticket`, not a correct one.
- **Put decision rules in the instructions.** It works on every provider.
- **Test classifiers on known cases.** A classifier that always answers "1" still
  produces perfectly valid objects.

### Other output types

`output_type` isn't limited to Pydantic models: `list[str]`, dataclasses,
`TypedDict`s and plain types all work:

In [ ]:
lister = Agent(name='Lister', instructions='Name three risks of sidewalk robots, a few words each.',
               model=model, output_type=list[str])
risks = (await Runner.run(lister, 'Go.')).final_output
print(type(risks).__name__, risks)

### Strict schemas

By default the SDK uses **strict** JSON schemas: every field required, no
open-ended `dict`s. That's what makes the output reliable. A type the strict
format can't express is rejected up front:

In [ ]:
class Profile(BaseModel):
    city: str
    facts: dict                       # arbitrary keys: not expressible in a strict schema

try:
    Agent(name='P', instructions='Describe a city.', model=model, output_type=Profile)
    await Runner.run(Agent(name='P', instructions='Describe Pune.', model=model, output_type=Profile), 'Go.')
except UserError as err:
    print('rejected:', str(err)[:160])

loose = Agent(name='P', instructions='Give 3 facts about Pune as key/value pairs.', model=model,
              output_type=AgentOutputSchema(Profile, strict_json_schema=False))
print('non-strict:', (await Runner.run(loose, 'Go.')).final_output)

`AgentOutputSchema(..., strict_json_schema=False)` opts out, at the cost of
weaker guarantees. Usually the better fix is to **make the type strict**, for
example a list of `{key, value}` objects instead of a free-form dict.

Also look at the facts themselves. The output parsed perfectly, but a model
answering from memory can still invent details. A valid structure is not
evidence of true content; grounding comes from tools and documents (notebooks 04 and 11).

## 5 · Counting tokens

Every run records token usage in `result.context_wrapper.usage`, summed over
all model calls in the run, with a per-request breakdown:

In [ ]:
usage = result.context_wrapper.usage
print(f'requests={usage.requests}  input={usage.input_tokens}  output={usage.output_tokens}  total={usage.total_tokens}')
for i, req in enumerate(usage.request_usage_entries, 1):
    print(f'  call {i}: {req.input_tokens} in / {req.output_tokens} out')

Multiply by your provider's price per million tokens for a cost estimate
(Ollama: $0). Notebook 12 turns this into a budget.

## 6 · Trade-offs

| | Local (Ollama) | Cloud (OpenAI) |
|---|---|---|
| Cost | free after hardware | per token |
| Privacy | nothing leaves your machine | data sent to a provider |
| API features | Chat Completions only; `tool_choice` ignored | Responses API, hosted tools, full settings |
| Quality on hard tasks | good for mid-size models | strongest models |

Structured output is almost always worth it for anything code will act on. The
cost is a less flexible response, and on small models an occasional validation
failure (`ModelBehaviorError`, handled in notebook 12).

## 7 · Common mistakes

- **Decision rules only in field descriptions.** The model may never read them. Put rubrics in the instructions.
- **Treating a valid object as a correct one.** Test on known cases, and ground facts with tools.
- **Open-ended `dict` fields** in `output_type`. Make the type strict instead of switching strict mode off.
- **Hard-coding a model string everywhere.** Use one helper (like `get_model`) or `RunConfig`.
- **Forgetting tracing** when running without an OpenAI key. Disable export, or record locally (notebook 10).

## 8 · Check yourself

<details><summary><b>1.</b> Why does this section use <code>OpenAIChatCompletionsModel</code> for Ollama?</summary>

Ollama implements an OpenAI-compatible Chat Completions endpoint, not the Responses API. `OpenAIChatCompletionsModel` speaks Chat Completions through any `AsyncOpenAI` client, so pointing that client at Ollama's `/v1` URL works.
</details>

<details><summary><b>2.</b> What exactly does <code>output_type</code> guarantee, and what doesn't it?</summary>

It guarantees the final output is valid for the type (fields present, types right, `Literal` values allowed, constraints met), or the run fails. It doesn't guarantee the values are correct.
</details>

<details><summary><b>3.</b> Your classifier's urgency scale is in a <code>Field(description=...)</code> and results look random. What's the likely cause and fix?</summary>

On providers like Ollama, the schema constrains the output format but isn't read as instructions, so the model never sees the scale. Put the rubric in the agent's instructions.
</details>

<details><summary><b>4.</b> Why is strict schema mode the default, and when would you turn it off?</summary>

Strict schemas make every field required and forbid open-ended objects, so outputs are reliable and fully typed. Turn it off (`AgentOutputSchema(..., strict_json_schema=False)`) only when a type genuinely can't be expressed strictly, and prefer reshaping the type instead.
</details>

<details><summary><b>5.</b> How would you give a router a cheap model and a writer a strong one?</summary>

Give each `Agent` its own `model` (e.g. `get_model('ollama')` for the router, `get_model('openai')` for the writer). Models are per agent.
</details>

## Takeaway

Models are pluggable: a string for OpenAI, a `Model` object for anything else
(Ollama via `OpenAIChatCompletionsModel`). Tune them with `ModelSettings`, and
use **`output_type`** whenever code will act on the answer. But put your decision
rules in the instructions, because a valid object isn't necessarily a correct one.

Next → `04_tools.ipynb`: writing good tools, handling their errors, and
controlling what happens after a tool runs.